Thanks to both of you for your inputs today!  Attached are the DeepSTARR paper and supplement, and below is the current list of fields for the input file, and an excerpt from the paper.
 
```
HepG2   chr9:102223-102492   RefSequence   log2FC   (reps:DNA-counts,...)  (reps:RNA-counts,...)   CradleBiasFactor
``` 
Best,


while Kari works on finding the right files to use here, tryin stictching together the code to count in windows, using a sliding window approach


In [3]:
%%bash
mkdir -p /data/reddylab/Alex/collab/20221017_Bill/results
mkdir -p /data/reddylab/Alex/collab/20221017_Bill/data
mkdir -p /data/reddylab/Alex/collab/20221017_Bill/logs


In [1]:
%%bash
export LD_LIBRARY_PATH="/data/reddylab/Alex/software/miniforge3/envs/alex_py3/lib/:${LD_LIBRARY_PATH}"
source /data/reddylab/Alex/software/miniforge3/bin/activate alex_py3

sbatch \
    --account=reddylab \
    --mem=8G \
    --array=1,2,3,4 \
    --cpus-per-task=1 \
    -o logs/bwtool.window_counts_600_50.%a.out \
    <<'EOF'
#!/bin/bash
# ,/data/reddylab/kstrouse/superstarr/input_libs/A001/nextseq/processing/starr_seq/A001_nextseq-pe/rep3.f3q10.sorted.dedup.rpkm.bw,/data/reddylab/kstrouse/superstarr/input_libs/A001/nextseq/processing/starr_seq/A001_nextseq-pe/rep4.f3q10.sorted.dedup.rpkm.bw
bwtool window 600 \
    /data/reddylab/kstrouse/superstarr/input_libs/A001/nextseq/processing/starr_seq/A001_nextseq-pe/rep${SLURM_ARRAY_TASK_ID}.f3q10.sorted.dedup.raw.bw \
    -step=50 \
    -skip-NA \
| awk -vOFS="\t" 'NR==1{ split($4, vv, ","); nwins=length(vv)}{tot=0; split($4, vv, ","); for (ii=1; ii<=nwins; ii+=1){tot+=vv[ii]} print $1,$2,$3,tot/nwins}' \
| awk '$4!=0' \
> /data/reddylab/Alex/collab/20221017_Bill/data/rep${SLURM_ARRAY_TASK_ID}.f3q10.sorted.dedup.w600s50.bdg
EOF

Submitted batch job 120185


In [3]:
%%bash
export LD_LIBRARY_PATH="/data/reddylab/Alex/software/miniforge3/envs/alex_py3/lib/:${LD_LIBRARY_PATH}"
source /data/reddylab/Alex/software/miniforge3/bin/activate alex_py3

sbatch \
    --account=reddylab \
    --mem=8G \
    --array=1 \
    --cpus-per-task=1 \
    -o logs/bwtool.window_counts_600_50.outputs.%a.out \
    <<'EOF'
#!/bin/bash
# ,/data/reddylab/kstrouse/superstarr/input_libs/A001/nextseq/processing/starr_seq/A001_nextseq-pe/rep3.f3q10.sorted.dedup.rpkm.bw,/data/reddylab/kstrouse/superstarr/input_libs/A001/nextseq/processing/starr_seq/A001_nextseq-pe/rep4.f3q10.sorted.dedup.rpkm.bw
#     /data/reddylab/kstrouse/superstarr/output_libs/A001_K562/A001_K562_20201124/nextseq/processing/starr_seq/A001_K562_20201124_nextseq-pe/A001-K562-rep${SLURM_ARRAY_TASK_ID}-nextseq.f3q10.sorted.dedup.raw.bw \
    # /data/reddylab/kstrouse/superstarr/output_libs/A001_K562/A001_K562_20210213/processing/starr_seq/Strouse_6825_210223A5-pe/A001-K562-rep${SLURM_ARRAY_TASK_ID}.f3q10.sorted.dedup.raw.bw \
bwtool window 600 \
    /data/reddylab/kstrouse/superstarr/output_libs/A001_K562/A001_K562_20201124/nextseq/processing/starr_seq/A001_K562_20201124_nextseq-pe/A001-K562-rep${SLURM_ARRAY_TASK_ID}-nextseq.f3q10.sorted.dedup.raw.bw \
    -step=50 \
    -skip-NA \
| awk -vOFS="\t" 'NR==1{ split($4, vv, ","); nwins=length(vv)}{tot=0; split($4, vv, ","); for (ii=1; ii<=nwins; ii+=1){tot+=vv[ii]} print $1,$2,$3,tot/nwins}' \
| awk '$4!=0' \
> /data/reddylab/Alex/collab/20221017_Bill/data/A001-K562-rep${SLURM_ARRAY_TASK_ID}-nextseq.f3q10.sorted.dedup.w600s50.bdg
EOF

Submitted batch job 120191


In [2]:
(3+3+3+3+2+2+2+2+1+2)/9.

2.5555555555555554

In [2]:
# %load_ext memory_profiler
# def lol(x):
#     return x
# %memit lol(500)


In [ ]:
# import os
# import numpy as np
# import pandas as pd
# import pathlib

# data_dir = '/data/reddylab/Alex/collab/20221017_Bill/data'

# # First, find all common fragments in the inputs
# df = None
# for f in os.listdir(data_dir):
#     if f.endswith('.w600s50.bdg') and ('A001-K562' in f) :
#         print(f)
#         df_tmp = pd.read_csv(os.path.join(data_dir, f),
#                          sep='\t', 
#                          names = ['chrom', 'start', 'end', 'count'])
#         df_tmp.index = df_tmp['chrom'] + "_" + df_tmp['start'].astype('str') + "_" + df_tmp['end'].astype('str')
#         if df is None:
#             df = df_tmp.index
#         else:
#             df = df.join(df_tmp.index, how='inner')
# input_df = df.copy()

# # for treatment in ['DMSO', 'Dex']:
# for treatment in ["AZD2906", 'AZD9567','CORT108297','CpdA','GW870086','Hydrocortisone','Mapracorat','RU486','ZK216348']: #'AZD2906',
#     outdir = f'{data_dir}/by_treatment/{treatment}'

#     df = None
#     for f in os.listdir(data_dir):
#         if f.endswith('.w300s50.bdg') and (treatment in f) :
#             print(f)
#             df_tmp = pd.read_csv(os.path.join(data_dir, f),
#                              sep='\t', 
#                              names = ['chrom', 'start', 'end', 'count'])
#             df_tmp.index = df_tmp['chrom'] + "_" + df_tmp['start'].astype('str') + "_" + df_tmp['end'].astype('str')
#             if df is None:
#                 df = df_tmp.index
#             else:
#                 df = df.join(df_tmp.index, how='inner')

#     # merge with inputs
#     df = df.join(input_df, how='inner')
    
#     pathlib.Path(outdir).mkdir(exist_ok=True)
#     for f in os.listdir(data_dir):
#         if f.endswith('.w300s50.bdg') and (treatment in f or 'Input' in f) :
#             print(f)
#             df_tmp = pd.read_csv(os.path.join(data_dir, f),
#                              sep='\t', 
#                              names = ['chrom', 'start', 'end', 'count'])
#             df_tmp.index = df_tmp['chrom'] + "_" + df_tmp['start'].astype('str') + "_" + df_tmp['end'].astype('str')
#             df_tmp.join(pd.DataFrame(index=df), how='inner')\
#                 .to_csv(os.path.join(outdir, f.replace('.bdg', '.in_common_win.bdg')), 
#                         sep='\t', index = False)


Find set of windows with data for all replicates/conditions

In [ ]:
import numpy as np
import pandas as pd
df = pd.read_csv('/data/reddylab/Alex/collab/20221017_Bill/data/rep2.f3q10.sorted.dedup.w600s50.bdg', sep='\t', names = ['chrom', 'start', 'end', 'count'])
df.index = df['chrom'] + "_" + df['start'].astype('str') + "_" + df['end'].astype('str')
df_tmp = pd.read_csv('/data/reddylab/Alex/collab/20221017_Bill/data/rep3.f3q10.sorted.dedup.w600s50.bdg', sep='\t', names = ['chrom', 'start', 'end', 'count'])
df_tmp.index = df_tmp['chrom'] + "_" + df_tmp['start'].astype('str') + "_" + df_tmp['end'].astype('str')
df = df.index.join(df_tmp.index, how='inner')
df_tmp = pd.read_csv('/data/reddylab/Alex/collab/20221017_Bill/data/rep4.f3q10.sorted.dedup.w600s50.bdg', sep='\t', names = ['chrom', 'start', 'end', 'count'])
df_tmp.index = df_tmp['chrom'] + "_" + df_tmp['start'].astype('str') + "_" + df_tmp['end'].astype('str')
df = df.join(df_tmp.index, how='inner')



In [ ]:
# import numpy as np
# import pandas as pd
# bedgraphs = [
#     '/data/reddylab/Alex/collab/20221017_Bill/data/A001-K562-rep2-nextseq.f3q10.sorted.dedup.w600s50.bdg', 
#     '/data/reddylab/Alex/collab/20221017_Bill/data/A001-K562-rep3-nextseq.f3q10.sorted.dedup.w600s50.bdg', 
#     '/data/reddylab/Alex/collab/20221017_Bill/data/A001-K562-rep4-nextseq.f3q10.sorted.dedup.w600s50.bdg' 
# ]
# # df = pd.read_csv(bedgraphs[0], sep='\t', names = ['chrom', 'start', 'end', 'count'])
# # df.index = df['chrom'] + "_" + df['start'].astype('str') + "_" + df['end'].astype('str')
# for ii in bedgraphs[1:]:
#     df_tmp = pd.read_csv(ii, sep='\t', names = ['chrom', 'start', 'end', 'count'])
#     df_tmp.index = df_tmp['chrom'] + "_" + df_tmp['start'].astype('str') + "_" + df_tmp['end'].astype('str')
#     df = df.index.join(df_tmp.index, how='inner')


In [ ]:
for rep in [1,2,3]:
    df_tmp = pd.read_csv(f'/data/reddylab/Alex/collab/20221017_Bill/data/A001-K562-rep{rep}-nextseq.f3q10.sorted.dedup.w600s50.bdg', sep='\t', names = ['chrom', 'start', 'end', 'count'])
    df_tmp.index = df_tmp['chrom'] + "_" + df_tmp['start'].astype('str') + "_" + df_tmp['end'].astype('str')
    df_tmp.join(pd.DataFrame(index=df), how='inner')\
        .to_csv(f'/data/reddylab/Alex/collab/20221017_Bill/data/A001-K562-rep{rep}-nextseq.f3q10.sorted.dedup.w600s50.in_common_win.bdg', 
                sep='\t', index = False)


In [ ]:
for rep in [2,3,4]:
    df_tmp = pd.read_csv(f'/data/reddylab/Alex/collab/20221017_Bill/data/rep{rep}.f3q10.sorted.dedup.w600s50.bdg', sep='\t', names = ['chrom', 'start', 'end', 'count'])
    df_tmp.index = df_tmp['chrom'] + "_" + df_tmp['start'].astype('str') + "_" + df_tmp['end'].astype('str')
    df_tmp.join(pd.DataFrame(index=df), how='inner')\
        .to_csv(f'/data/reddylab/Alex/collab/20221017_Bill/data/rep{rep}.f3q10.sorted.dedup.w600s50.in_common_win.bdg', 
                sep='\t', index = False)


In [ ]:
%%bash
for THRES in 100 150 200;
do
    paste -d"\t" \
        <(cat /data/reddylab/Alex/collab/20221017_Bill/data/rep2.f3q10.sorted.dedup.w600s50.in_common_win.bdg) \
        <(cat /data/reddylab/Alex/collab/20221017_Bill/data/rep3.f3q10.sorted.dedup.w600s50.in_common_win.bdg) \
        <(cat /data/reddylab/Alex/collab/20221017_Bill/data/rep4.f3q10.sorted.dedup.w600s50.in_common_win.bdg) \
        <(cat /data/reddylab/Alex/collab/20221017_Bill/data/A001-K562-rep1-nextseq.f3q10.sorted.dedup.w600s50.in_common_win.bdg) \
        <(cat /data/reddylab/Alex/collab/20221017_Bill/data/A001-K562-rep2-nextseq.f3q10.sorted.dedup.w600s50.in_common_win.bdg) \
        <(cat /data/reddylab/Alex/collab/20221017_Bill/data/A001-K562-rep3-nextseq.f3q10.sorted.dedup.w600s50.in_common_win.bdg) \
    | awk -vTHRES=${THRES} -vOFS="\t" 'BEGIN{print "chrom\tstart\tend\tinput_rep1\tinput_rep2\tinput_rep3\toutput_rep1\toutput_rep2\toutput_rep3"}($4+$8+$12>THRES) && ($16+$20+$24>THRES){print $1,$2,$3,$4,$8,$12,$16,$20,$24}' \
    > /data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_${THRES}.bdg
done

In [ ]:
%%bash
for THRES in 100 150 200;
do
    paste -d"\t" \
        <(cat /data/reddylab/Alex/collab/20221017_Bill/data/rep2.f3q10.sorted.dedup.w600s50.in_common_win.bdg) \
        <(cat /data/reddylab/Alex/collab/20221017_Bill/data/rep3.f3q10.sorted.dedup.w600s50.in_common_win.bdg) \
        <(cat /data/reddylab/Alex/collab/20221017_Bill/data/rep4.f3q10.sorted.dedup.w600s50.in_common_win.bdg) \
        <(cat /data/reddylab/Alex/collab/20221017_Bill/data/A001-K562-rep1-nextseq.f3q10.sorted.dedup.w600s50.in_common_win.bdg) \
        <(cat /data/reddylab/Alex/collab/20221017_Bill/data/A001-K562-rep2-nextseq.f3q10.sorted.dedup.w600s50.in_common_win.bdg) \
        <(cat /data/reddylab/Alex/collab/20221017_Bill/data/A001-K562-rep3-nextseq.f3q10.sorted.dedup.w600s50.in_common_win.bdg) \
    | awk -vTHRES=${THRES} -vOFS="\t" 'BEGIN{print "chrom\tstart\tend\tinput_rep1\tinput_rep2\tinput_rep3\toutput_rep1\toutput_rep2\toutput_rep3"}($4+$8+$12>THRES){print $1,$2,$3,$4,$8,$12,$16,$20,$24}' \
    > /data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_${THRES}.thres_dna_only.bdg
done

Add lib.sizes

In [3]:
%%bash
cat /data/reddylab/kstrouse/superstarr/input_libs/A001/nextseq/processing/starr_seq/A001_nextseq-pe/rep{2,3,4}*.R1.trimmed.read_count.txt > /data/reddylab/Alex/collab/20221017_Bill/data/K562.starrseq.A001_combined.lib_sizes.txt.tmp
cat /data/reddylab/kstrouse/superstarr/output_libs/A001_K562/A001_K562_20201124/nextseq/processing/starr_seq/A001_K562_20201124_nextseq-pe/A001-K562-rep*.R1.trimmed.read_count.txt /data/reddylab/kstrouse/superstarr/output_libs/A001_K562/A001_K562_20210213/processing/starr_seq/Strouse_6825_210223A5-pe/A001-K562-rep*.R1.trimmed.read_count.txt >> /data/reddylab/Alex/collab/20221017_Bill/data/K562.starrseq.A001_combined.lib_sizes.txt.tmp
paste -d"\t" \
    <(head -n1 /data/reddylab/Alex/collab/20221017_Bill/data/combined.input_and_output.gt_100.bdg  | cut -f4- | tr '\t' '\n') \
    /data/reddylab/Alex/collab/20221017_Bill/data/K562.starrseq.A001_combined.lib_sizes.txt.tmp \
> /data/reddylab/Alex/collab/20221017_Bill/data/K562.starrseq.A001_combined.lib_sizes.txt \
&& rm -f /data/reddylab/Alex/collab/20221017_Bill/data/K562.starrseq.A001_combined.lib_sizes.txt.tmp 

In [1]:
1.49459e+09 +1

1494590001.0

In [1]:
import numpy as np

import pandas as pd
for thres in [100, 150, 200]:
    df = pd.read_csv(f'/data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_{thres}.bdg', sep='\t')
#     df.astype(int, errors='ignore')\
#         .to_csv(f'/data/reddylab/Alex/collab/20221017_Bill/data/combined.input_and_output.gt_{thres}.txt', index=False, sep='\t')
    df = df.astype(int, errors='ignore')
    df.iloc[:, 3:] = df.iloc[:, 3:]/df.iloc[:, 3:].sum()*1e6
    
    log2FC_tmp = (np.log2((0.001+df.iloc[:, 6:].mean(axis=1)) / 
           (0.001+df.iloc[:, 3:6].mean(axis=1))))
    df = None
    df = pd.read_csv(f'/data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_{thres}.bdg', sep='\t')
    df = df.astype(int, errors='ignore')
    df['log2FC'] = log2FC_tmp
    df.to_csv(f'/data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_{thres}.log2FC.txt', 
              sep='\t', index=False, float_format='%.5f')

In [2]:
import numpy as np

import pandas as pd
for thres in [100, 150, 200]:
    df = pd.read_csv(f'/data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_{thres}.thres_dna_only.bdg', sep='\t')
#     df.astype(int, errors='ignore')\
#         .to_csv(f'/data/reddylab/Alex/collab/20221017_Bill/data/combined.input_and_output.gt_{thres}.txt', index=False, sep='\t')
    df = df.astype(int, errors='ignore')
    df.iloc[:, 3:] = df.iloc[:, 3:]/df.iloc[:, 3:].sum()*1e6
    
    log2FC_tmp = (np.log2((0.001+df.iloc[:, 6:].mean(axis=1)) / 
           (0.001+df.iloc[:, 3:6].mean(axis=1))))
    df = None
    df = pd.read_csv(f'/data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_{thres}.thres_dna_only.bdg', sep='\t')
    df = df.astype(int, errors='ignore')
    df['log2FC'] = log2FC_tmp
    df.to_csv(f'/data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_{thres}.thres_dna_only.log2FC.txt', 
              sep='\t', index=False, float_format='%.5f')

In [ ]:
awk -vOFS="\t" 'NR==1{nwins=length(vv)}{tot=0; split($4, vv, ","); for (ii=1; ii<=nwins; ii+=1){tot+=vv[ii]} print $1,$2,$3,tot/nwins}' \


In [3]:
%%bash
for THRES in 100 150 200;
do
    tail -n+2 /data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_${THRES}.log2FC.txt \
    | cut -f1-3 \
    | bedtools getfasta \
        -fi /data/reddylab/Reference_Data/Genomes/hg38/GCA_000001405.15_GRCh38_full_analysis_set.fna \
        -bed stdin \
    > /data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_${THRES}.sequences.fa

    tail -n+2 /data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_${THRES}.thres_dna_only.log2FC.txt \
    | cut -f1-3 \
    | bedtools getfasta \
        -fi /data/reddylab/Reference_Data/Genomes/hg38/GCA_000001405.15_GRCh38_full_analysis_set.fna \
        -bed stdin \
    > /data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_${THRES}.thres_dna_only.sequences.fa

done

In [4]:
%%bash
for THRES in 100 150 200;
do
    paste -d"\t" /data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_${THRES}.log2FC.txt \
        <(cat <(echo sequence) <(cat /data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_${THRES}.sequences.fa | awk 'NR%2==0')) \
    | gzip -c \
        > /data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_${THRES}.log2FC.sequence.txt.gz
done

In [5]:
%%bash
for THRES in 100 150 200;
do
    paste -d"\t" /data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_${THRES}.thres_dna_only.log2FC.txt \
        <(cat <(echo sequence) <(cat /data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_${THRES}.thres_dna_only.sequences.fa | awk 'NR%2==0')) \
    | gzip -c \
        > /data/reddylab/Alex/collab/20221017_Bill/data/K562.combined.input_and_output.w600s50.gt_${THRES}.thres_dna_only.log2FC.sequence.txt.gz
done

In [ ]:
bwtool window 300 /data/reddylab/Alex/encode4_duke/ipynbs/jamborees/20211206_MPRA_STARR_Jamboree/data/K562_GROcap_mn.bw 
--step=100000 -skip-NA | head -n5 |  awk -vOFS="\t" 'NR==1{ split($4, vv, ","); nwins=length(vv)}{tot=0; split($4, vv, ","); for (ii=1; ii<=nwins; ii+=1){tot+=vv[ii]} print $1,$2,$3,tot/nwins}'

**22-11-08 Notes**
Bill would like to compare the log2FC values I calculated (CPMs) with those computed with DESeq2. Because it would take a minute to run DESeq2 on the entire dataset, try using only one chromosome. 

In [35]:
%%bash
THRES=200
CHROM="chr9"

for THRES in 100 150;
do
    echo -e "peak\twgstarr.A001.input.rep1\twgstarr.A001.input.rep2\twgstarr.A001.input.rep3\twgstarr.A001.output.rep1\twgstarr.A001.output.rep2\twgstarr.A001.output.rep3" \
        > /data/reddylab/Alex/collab/20221017_Bill/data/combined.input_and_output.gt_${THRES}.counts.${CHROM}.txt
    gzip -dc /data/reddylab/Alex/collab/20221017_Bill/data/combined.input_and_output.gt_${THRES}.log2FC.sequence.txt.gz \
    | /bin/grep -wP "^${CHROM}" \
    | awk -vOFS="\t" '{print $1"_"$2"_"$3, $4,$5,$6,$7,$8,$9}' \
    >> /data/reddylab/Alex/collab/20221017_Bill/data/combined.input_and_output.gt_${THRES}.counts.${CHROM}.txt

#     gzip -dc /data/reddylab/Alex/collab/20221017_Bill/data/combined.input_and_output.gt_${THRES}.log2FC.sequence.txt.gz \
#     | /bin/grep -wP "^${CHROM}" \
#     | awk -vOFS="\t" '{print $1"_"$2"_"$3,$10}' \
#     > /data/reddylab/Alex/collab/20221017_Bill/data/combined.input_and_output.gt_${THRES}.cpm_log2fc.${CHROM}.txt
done